# 03 - Bronze Data Profiling

Profiles the six Bronze source tables without modifying their data.

Saves the profiling results in one table: `bronze_profiling_summary` (overwritten each run with the current profile, so re-runs are idempotent).

Checks:
- Total row count
- Null count for each column
- Distinct count for each column
- Duplicate business-key values

Run this notebook after the JDBC and ADLS notebooks. Bronze validation checks (including region consistency) are in `tests/06_Pipeline_Validation`, which saves `bronze_validation_summary`.

In [0]:
# 1. Configuration
catalog = "capstone6_telecom"
schema = "bronze"

tables = {
    "bronze_tower_master": "tower_id",
    "bronze_network_topology_master": "cell_id",
    "bronze_region_lookup": "region_id",
    "bronze_network_kpi_measurements": "measurement_id",
    "bronze_network_outage_events": "outage_id",
    "bronze_customer_complaints": "complaint_id"
}

In [0]:
# 2. Create the profiling results table if it does not exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.bronze_profiling_summary (
    table_name STRING,
    column_name STRING,
    total_rows BIGINT,
    null_count BIGINT,
    distinct_count BIGINT,
    duplicate_key_values BIGINT,
    profiled_at TIMESTAMP
)
USING DELTA
""")

In [0]:
# 3. Profile the six Bronze tables
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType
)

results = []

for table_name, key_column in tables.items():

    df = spark.table(f"{catalog}.{schema}.{table_name}")
    total_rows = df.count()

    # Count how many business-key values appear more than once
    duplicate_key_values = (df.groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count())

    # Add one table-level summary row
    results.append((
        table_name,
        "ALL_COLUMNS",
        total_rows,
        None,
        None,
        duplicate_key_values
    ))

    # Add one row for each column
    for column_name in df.columns:

        stats = df.agg(
            F.sum(
                F.when(F.col(column_name).isNull(), 1).otherwise(0)
            ).alias("nulls"),
            F.countDistinct(F.col(column_name)).alias("distincts")
        ).first()

        results.append((
            table_name,
            column_name,
            total_rows,
            int(stats["nulls"] or 0),
            int(stats["distincts"] or 0),
            None
        ))

print("Profiling completed.")

In [0]:
# 4. Save results to bronze_profiling_summary
result_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("column_name", StringType(), False),
    StructField("total_rows", LongType(), True),
    StructField("null_count", LongType(), True),
    StructField("distinct_count", LongType(), True),
    StructField("duplicate_key_values", LongType(), True)
])

result_df = spark.createDataFrame(results, schema=result_schema)
result_df = result_df.withColumn("profiled_at", F.current_timestamp())

# Overwrite: the table holds the current profile only, so re-running does not add duplicate profile rows
(result_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema}.bronze_profiling_summary"))

display(result_df.orderBy("table_name", "column_name"))

In [0]:
# 5. View the saved profiling results
display(
    spark.table(f"{catalog}.{schema}.bronze_profiling_summary")
    .orderBy("table_name", "column_name")
)